# Problem Statement
* You are building an AI tool for a recruiter:
* Input → Resume + Job Description
* Process → Skill Extraction + Matching + Scoring
* Output → Fit Score + Explanation


In [41]:
# Installing Required packages
import sys
!{sys.executable} -m pip install langchain langchain-groq langsmith pymupdf python-dotenv
print("All packages installed.")

All packages installed.


# Load env and API Keys

In [42]:
from dotenv import load_dotenv
load_dotenv()

import os

# Groq API key — used to call the free LLaMA 3 model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# ─────────────────────────────────────────────────────────────
# LangSmith tracing configuration
# LANGCHAIN_TRACING_V2=true  → enables automatic run tracing
# LANGCHAIN_PROJECT          → groups all runs under this project name
# ─────────────────────────────────────────────────────────────
os.environ["LANGCHAIN_TRACING_V2"]  = "true"
os.environ["LANGCHAIN_API_KEY"]     = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"]     = "AI-Resume-Screener-Groq"

print("API keys configured.")
print(f"   Groq model      : llama-3.1-8b-instant(free)")
print(f"   LangSmith project: {os.environ['LANGCHAIN_PROJECT']}")
print(f"   Tracing enabled  : {os.environ['LANGCHAIN_TRACING_V2']}")

API keys configured.
   Groq model      : llama-3.1-8b-instant(free)
   LangSmith project: AI-Resume-Screener-Groq
   Tracing enabled  : true


# PDF text Extractor 

In [43]:
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract plain text from a PDF file using PyMuPDF.
    
    Args:
        pdf_path : Full path to the PDF file
    
    Returns:
        str : All text content from the PDF, pages joined by newline
    
    Raises:
        FileNotFoundError if the PDF path doesn't exist
    """
    # Check if file exists 
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found: {pdf_path}")
    
    extracted_pages = []  # Store text from each page
    
    # Open the PDF document
    doc = fitz.open(pdf_path)
    
    # Loop through each page and extract text
    for page_num in range(len(doc)):
        page = doc[page_num]               # Get page object
        text = page.get_text("text")       # Extract as plain text
        extracted_pages.append(text)       # Collect each page's text
    
    doc.close()  # Always close the document after reading
    
    # Join all pages into one string, separated by newlines
    full_text = "\n".join(extracted_pages)
    
    # Basic cleanup — remove excessive blank lines
    full_text = "\n".join(
        line for line in full_text.splitlines() if line.strip()
    )
    
    return full_text

# Load PDF Resume

In [44]:
# ── PDF file paths ────
PDF_PATHS = {
    "Strong Candidate"  : "strong_resume.pdf",   # Replace with your file
    "Average Candidate" : "average_resume.pdf",  # Replace with your file
    "Weak Candidate"    : "weak_resume.pdf",     # Replace with your file
}

# Extract text from actual PDF files using PyMuPDF
RESUMES = {}
for label, pdf_path in PDF_PATHS.items():
    try:
        # Extract raw text from the PDF
        resume_text = extract_text_from_pdf(pdf_path)
            
        # Use filename (without extension) as candidate name
        candidate_name = os.path.splitext(os.path.basename(pdf_path))[0]
            
        RESUMES[candidate_name] = (label, resume_text)
        print(f"✅ Loaded: {pdf_path} ({len(resume_text)} characters)")
        
    except FileNotFoundError as e:
        # Warn but don't crash — Message missing files
        print(f"⚠️  : {e}")

✅ Loaded: strong_resume.pdf (2007 characters)
✅ Loaded: average_resume.pdf (364 characters)
✅ Loaded: weak_resume.pdf (157 characters)


# Job description

In [45]:
# ─────────────────────────────────────────────────────────────
# JOB DESCRIPTION
# This is the reference document. All resumes are scored
# relative to these requirements.
# You can swap this out for any job description.
# ─────────────────────────────────────────────────────────────
JOB_DESCRIPTION = """
Job Title: Senior Data Scientist
Company: TechCorp Analytics

Required Skills:
- Programming: Python (advanced), SQL (intermediate)
- Machine Learning: Scikit-learn, XGBoost, model evaluation, cross-validation
- Deep Learning: TensorFlow or PyTorch
- Data Processing: Pandas, NumPy, feature engineering
- Cloud Platforms: AWS (SageMaker preferred) or GCP/Azure
- MLOps: MLflow or similar experiment tracking
- Visualization: Matplotlib, Seaborn, or Tableau
- Statistics: Hypothesis testing, A/B testing, regression analysis
- Version Control: Git / GitHub

Nice-to-Have:
- NLP experience (transformers, BERT, LLMs)
- Big Data tools: Spark, Hadoop
- Docker / Kubernetes
- Published research or Kaggle achievements

Experience: Minimum 3 years of industry Data Scientist experience
Education: Bachelor's or Master's in CS, Statistics, Mathematics, or related field
"""

print("Job Description loaded: Senior Data Scientist @ TechCorp Analytics")

Job Description loaded: Senior Data Scientist @ TechCorp Analytics


# Prompt Template
* Prompt with anti-Hallucination rules,output format constraints, and few shot prompting 

In [46]:
from langchain_core.prompts import PromptTemplate

# ═══════════════════════════════════════════════════════════════
# PROMPT TEMPLATES
# All prompts follow 3 principles:
#   1. Clear role + task description
#   2. Strict anti-hallucination rules (only use what's in resume)
#   3. Constrained output format (JSON where applicable)
# ═══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
# PROMPT 1: SKILL EXTRACTION
# Extracts structured fields from the raw resume text.
# Key anti-hallucination rule: only extract what's explicitly stated.
# ─────────────────────────────────────────────────────────────
extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are a precise resume parser.

STRICT RULES:
- Only extract information EXPLICITLY written in the resume.
- Do NOT infer or assume any skills not present in the resume text.
- If a field has no data in the resume, write "Not mentioned".

Extract these fields from the resume below:
1. programming_languages : e.g. ["Python", "SQL"]
2. ml_frameworks         : e.g. ["Scikit-learn", "TensorFlow"]
3. data_tools            : e.g. ["Pandas", "Spark"]
4. cloud_platforms       : e.g. ["AWS SageMaker", "GCP"]
5. mlops_tools           : e.g. ["MLflow", "Docker"]
6. visualization_tools   : e.g. ["Tableau", "Matplotlib"]
7. years_of_experience   : integer (professional DS/ML experience only)
8. education_level       : e.g. "M.Tech Computer Science — IIT Bombay"
9. notable_achievements  : e.g. ["Kaggle Master", "NeurIPS paper"]
10. domain_experience    : e.g. ["e-commerce", "fintech"]

Respond ONLY with valid JSON. No explanation, no markdown fences.

RESUME:
---
{resume}
---
"""
)

# ─────────────────────────────────────────────────────────────
# PROMPT 2: MATCHING
# Compares extracted skills against job requirements.
# Produces a structured gap analysis.
# ─────────────────────────────────────────────────────────────
matching_prompt = PromptTemplate(
    input_variables=["extracted_skills", "job_description"],
    template="""
You are a technical recruiter comparing a candidate profile against a job description.

STRICT RULES:
- Return STRICT JSON:
    {{
      "matched_required_skills": ["skill1", "skill2"],
      "missing_required_skills": ["skill3"]
    }}
- Only evaluate skills present in the extracted_skills JSON.
- Do NOT assume the candidate has skills not listed there.
- Be objective. Do not favor or penalize any candidate.

Produce a match report with these fields:
1. matched_required_skills  : required JD skills the candidate HAS
2. missing_required_skills  : required JD skills the candidate LACKS
3. matched_nice_to_have     : bonus skills the candidate HAS
4. experience_fit           : {{"meets_requirement": true/false, "detail": "..."}}
5. education_fit            : {{"meets_requirement": true/false, "detail": "..."}}
6. overall_strengths        : top 3 strengths for this role
7. overall_gaps             : top 3 critical gaps for this role

Respond ONLY with valid JSON. No explanation, no markdown fences.

JOB DESCRIPTION:
---
{job_description}
---

CANDIDATE EXTRACTED PROFILE:
---
{extracted_skills}
---
"""
)

# ─────────────────────────────────────────────────────────────
# PROMPT 3: SCORING
# Assigns a 0–100 score using a weighted rubric.
# Rubric breakdown:
#   required_skills (50) + experience (25) + education (10)
#   + nice_to_have (10)  + achievements (5)
# ─────────────────────────────────────────────────────────────
scoring_prompt = PromptTemplate(
    input_variables=["match_analysis", "extracted_skills"],
    template="""
You are scoring a candidate for a Senior Data Scientist role.

SCORING RUBRIC (must total exactly 100 points):
  required_skills_score  (0–50) : proportion of required skills matched
  experience_score       (0–25) : years of experience + production ML work
  education_score        (0–10) : relevance of educational background
  nice_to_have_score     (0–10) : bonus/nice-to-have skills matched
  achievements_score     (0–5)  : publications, rankings, notable work

STRICT RULES:
- All sub-scores must be integers.
- final_score MUST equal the exact sum of all sub-scores.
- Do NOT inflate scores. Be fair and consistent.

Respond ONLY with valid JSON. No explanation, no markdown fences:
{{
  "required_skills_score" : <0–50>,
  "experience_score"      : <0–25>,
  "education_score"       : <0–10>,
  "nice_to_have_score"    : <0–10>,
  "achievements_score"    : <0–5>,
  "final_score"           : <0–100>,
  "score_justifications" : {{
    "required_skills" : "one line reason",
    "experience"      : "one line reason",
    "education"       : "one line reason",
    "nice_to_have"    : "one line reason",
    "achievements"    : "one line reason"
  }}
}}

MATCH ANALYSIS:
---
{match_analysis}
---

EXTRACTED SKILLS:
---
{extracted_skills}
---
"""
)

# ─────────────────────────────────────────────────────────────
# PROMPT 4: EXPLANATION (Few-Shot Prompting)
# Generates a plain-English recruiter evaluation.
# Few-shot: 3 example outputs are shown so the model learns
# the exact tone, length, and format expected.
# ─────────────────────────────────────────────────────────────
explanation_prompt = PromptTemplate(
    input_variables=["candidate_name", "score_data", "match_analysis", "extracted_skills"],
    template="""
You are a senior technical recruiter writing candidate evaluation summaries.

── FEW-SHOT EXAMPLES (learn the style and format) ──────────────

EXAMPLE 1 — Strong Candidate (Score 87/100):
"Alex Chen is an excellent fit for the Senior Data Scientist role, scoring 87/100.
With 5 years at Google, Alex covers all required technical skills including Python,
TensorFlow, AWS SageMaker, and MLflow. NLP expertise and Kaggle Master rank are
strong differentiators. Minor gaps include limited Spark experience.
RECOMMENDATION: Strong Hire — proceed to technical interview immediately."

EXAMPLE 2 — Average Candidate (Score 51/100):
"Maria Lopez is a moderate fit, scoring 51/100. She has 3 years of experience
and solid Python and Scikit-learn skills but lacks deep learning (no TensorFlow
or PyTorch), MLOps tooling, and has limited cloud exposure. She could be a fit
for a junior variant of this role with upskilling support.
RECOMMENDATION: Maybe — consider for junior position or with mentoring plan."

EXAMPLE 3 — Weak Candidate (Score 17/100):
"John Davis is not a match for this Senior Data Scientist position, scoring 17/100.
Only beginner Python and basic ML coursework are present, with no SQL, cloud
platforms, TensorFlow, or MLOps tools. Non-technical background and 1 year of
accounting work do not align with this role.
RECOMMENDATION: Do Not Hire — profile does not meet minimum requirements."

─────────────────────────────────────────────────────────────────

NOW evaluate: {candidate_name}

SCORE DATA: {score_data}
MATCH ANALYSIS: {match_analysis}
EXTRACTED SKILLS: {extracted_skills}

Write 3–5 sentences. End with exactly one RECOMMENDATION line:
  RECOMMENDATION: Strong Hire / Hire / Maybe (condition) / Do Not Hire

Output ONLY the evaluation paragraph. No JSON. No extra text.
"""
)

print("__4 prompt templates defined__")
print("   1. extraction_prompt  — extracts skills/tools/experience")
print("   2. matching_prompt    — matches against JD requirements")
print("   3. scoring_prompt     — 0-100 score with weighted rubric")
print("   4. explanation_prompt — few-shot recruiter evaluation")

__4 prompt templates defined__
   1. extraction_prompt  — extracts skills/tools/experience
   2. matching_prompt    — matches against JD requirements
   3. scoring_prompt     — 0-100 score with weighted rubric
   4. explanation_prompt — few-shot recruiter evaluation


# LangChain LCEL Chains (Groq LLM)

In [47]:
import json
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(
    model="llama-3.1-8b-instant",   # Free Groq model
    temperature=0.0,           # 0 = consistent, reproducible scoring
    max_tokens=1500            # Enough for detailed JSON outputs
)

# StrOutputParser converts LangChain message objects → plain Python strings
str_parser = StrOutputParser()

def safe_parse_json(text: str) -> dict:
    """
    Safely parse JSON string from LLM output.
    Handles markdown fences (```json ... ```) gracefully.
    """
    cleaned = text.strip()

    if cleaned.startswith("```"):
        lines   = cleaned.split("\n")
        cleaned = "\n".join(lines[1:-1]).strip()  # Remove first and last lines
    
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        # Don't crash — return structured error dict for debugging
        return {"parse_error": str(e), "raw_output": text[:300]}

# ─────────────────────────────────────────────────────────────
# LCEL CHAINS — using | (pipe) operator
# Each chain = PromptTemplate | ChatGroq | StrOutputParser
# The | operator composes Runnables left-to-right:
#   1. Prompt formats the input into a message
#   2. LLM generates a response
#   3. Parser converts response to plain string
# ─────────────────────────────────────────────────────────────

# Chain 1: Resume → Structured skills JSON
extraction_chain = extraction_prompt | llm | str_parser

# Chain 2: Skills + JD → Match analysis JSON
matching_chain   = matching_prompt   | llm | str_parser

# Chain 3: Match analysis → Score breakdown JSON
scoring_chain    = scoring_prompt    | llm | str_parser

# Chain 4: All data → Plain English explanation + recommendation
explanation_chain = explanation_prompt | llm | str_parser


print("LCEL chains built using | pipe operator:")
print("   extraction_chain  = extraction_prompt  | llm | str_parser")
print("   matching_chain    = matching_prompt    | llm | str_parser")
print("   scoring_chain     = scoring_prompt     | llm | str_parser")
print("   explanation_chain = explanation_prompt | llm | str_parser")

LCEL chains built using | pipe operator:
   extraction_chain  = extraction_prompt  | llm | str_parser
   matching_chain    = matching_prompt    | llm | str_parser
   scoring_chain     = scoring_prompt     | llm | str_parser
   explanation_chain = explanation_prompt | llm | str_parser


# Core Pipeline Function
## Orchestrates all 4 steps and returns final score + label (Strong / Average / Weak).

In [48]:
from langsmith import traceable

def assign_label(score: int) -> str:
    """Convert numeric score to Strong / Average / Weak label."""
    if score >= 75:
        return "🟢 STRONG"
    elif score >= 45:
        return "🟡 AVERAGE"
    else:
        return "🔴 WEAK"

# ─────────────────────────────────────────────────────────────
# MAIN SCREENING PIPELINE
# @traceable → enables LangSmith to record this entire function
# as one run, with all 4 chain calls visible as nested steps.
#
# Each chain is called with .invoke() which is the standard
# LangChain LCEL method to execute a chain synchronously.
# ─────────────────────────────────────────────────────────────
@traceable(
    name="resume_screening_pipeline",
    tags=["resume-screening", "groq", "llama3", "data-scientist-jd"]
)

def screen_candidate(
    candidate_name : str,
    resume_text    : str,
    job_description: str,
    candidate_type : str = "Unknown"
) -> dict:
    """
    Full 4-step resume screening pipeline.

    Args:
        candidate_name  : Name shown in outputs and LangSmith traces
        resume_text     : Plain text extracted from PDF (or demo text)
        job_description : Target job description text
        candidate_type  : Metadata label (Strong/Average/Weak)

    Returns:
        dict with all step outputs + final_score + label
    """

    print(f"\n{'━'*60}")
    print(f"  Screening: {candidate_name} ({candidate_type})")
    print(f"{'━'*60}")

    # ─────────────────────────────────────────────────────────
    # STEP 1 — SKILL EXTRACTION
    # .invoke() sends the resume text through the extraction chain.
    # Returns a JSON string which we parse into a Python dict.
    # ─────────────────────────────────────────────────────────
    print("\n⏳ Step 1/4: Extracting skills...")

    raw_extracted   = extraction_chain.invoke({"resume": resume_text})
    extracted_skills = safe_parse_json(raw_extracted)

    print(f"Languages : {extracted_skills.get('programming_languages', [])}")
    print(f"ML Tools  : {extracted_skills.get('ml_frameworks', [])}")
    print(f"Cloud     : {extracted_skills.get('cloud_platforms', [])}")
    print(f"Exp Years : {extracted_skills.get('years_of_experience', 'N/A')}")

    # ─────────────────────────────────────────────────────────
    # STEP 2 — MATCHING
    # Pass the extracted skills (as JSON string) and JD to the
    # matching chain. Returns gap analysis as JSON.
    # ─────────────────────────────────────────────────────────
    print("\n⏳ Step 2/4: Matching against JD...")

    raw_match    = matching_chain.invoke({
        "extracted_skills" : json.dumps(extracted_skills, indent=2),
        "job_description"  : job_description
    })
    match_analysis = safe_parse_json(raw_match)
    
    matched = match_analysis.get("matched_required_skills", [])
    missing = match_analysis.get("missing_required_skills", [])
    # 🔥 Fix: ensure list
    if isinstance(matched, dict):
        matched = list(matched.keys())
    
    if isinstance(missing, dict):
        missing = list(missing.keys())
    print(f" ✅ Matched : {len(matched)} skills — {matched[:3]}")
    print(f" ❌ Missing : {len(missing)} skills — {missing[:3]}")

    # ─────────────────────────────────────────────────────────
    # STEP 3 — SCORING
    # Pass match analysis + extracted skills to get a 0–100 score.
    # Score breakdown: Skills(50) + Exp(25) + Edu(10) + NTH(10) + Ach(5)
    # ─────────────────────────────────────────────────────────
    print("\n⏳ Step 3/4: Computing score...")

    raw_score  = scoring_chain.invoke({
        "match_analysis"   : json.dumps(match_analysis, indent=2),
        "extracted_skills" : json.dumps(extracted_skills, indent=2)
    })
    score_data  = safe_parse_json(raw_score)
    final_score = score_data.get("final_score", 0)

    print(f"   ✅ Skills  : {score_data.get('required_skills_score', '?')}/50")
    print(f"   ✅ Exp     : {score_data.get('experience_score', '?')}/25")
    print(f"   ✅ Edu     : {score_data.get('education_score', '?')}/10")
    print(f"   ✅ Bonus   : {score_data.get('nice_to_have_score', '?')}/10")
    print(f"   ✅ Achiev  : {score_data.get('achievements_score', '?')}/5")
    print(f"   🎯 SCORE   : {final_score}/100")

    # ─────────────────────────────────────────────────────────
    # STEP 4 — EXPLANATION
    # Generate plain-English recruiter evaluation using few-shot
    # prompted explanation chain.
    # ─────────────────────────────────────────────────────────
    print("\n⏳ Step 4/4: Generating explanation...")

    explanation = explanation_chain.invoke({
        "candidate_name"   : candidate_name,
        "score_data"       : json.dumps(score_data, indent=2),
        "match_analysis"   : json.dumps(match_analysis, indent=2),
        "extracted_skills" : json.dumps(extracted_skills, indent=2)
    })

    # ─────────────────────────────────────────────────────────
    # LABEL ASSIGNMENT — rule-based, no LLM
    # Strong (>=75) / Average (>=45) / Weak (<45)
    # ─────────────────────────────────────────────────────────
    label = assign_label(final_score)

    print(f"\n{'─'*60}")
    print(f"  RESULT: {label} (Score: {final_score}/100)")
    print(f"{'─'*60}")
    print(f"  {explanation}")
    print(f"{'─'*60}")

    # Return all intermediate + final outputs
    return {
        "candidate_name"   : candidate_name,
        "candidate_type"   : candidate_type,
        "final_score"      : final_score,
        "label"            : label,
        "extracted_skills" : extracted_skills,
        "match_analysis"   : match_analysis,
        "score_data"       : score_data,
        "explanation"      : explanation
    }

print("screen_candidate() pipeline function ready.")
print(" Labels: >=75 → 🟢 STRONG | >=45 → 🟡 AVERAGE | <45 → 🔴 WEAK")

screen_candidate() pipeline function ready.
 Labels: >=75 → 🟢 STRONG | >=45 → 🟡 AVERAGE | <45 → 🔴 WEAK


## Run Screener on All 3 Candidates
Each run creates a separate trace in LangSmith.

In [49]:
# ─────────────────────────────────────────────────────────────
# MAIN EXECUTION
# Loops through all 3 resumes, runs the full pipeline for each.
# Results are stored in all_results dict for summary below.
# Each iteration = 1 LangSmith trace with 4 nested chain calls.
# ─────────────────────────────────────────────────────────────

all_results = {}  # Store all results here

print("🚀 Starting AI Resume Screener...")
print(f"   Job       : Senior Data Scientist @ TechCorp Analytics")
print(f"   Candidates: {len(RESUMES)}")
print(f"   LLM       : Groq / llama-3.1-8b-instant(free)")
print(f"   Tracing   : LangSmith — project '{os.environ['LANGCHAIN_PROJECT']}'")

# Process each candidate
for candidate_name, (candidate_type, resume_text) in RESUMES.items():

    # Run the full 4-step pipeline
    # Each call creates one top-level trace in LangSmith
    result = screen_candidate(
        candidate_name  = candidate_name,
        resume_text     = resume_text,
        job_description = JOB_DESCRIPTION,
        candidate_type  = candidate_type
    )

    # Store result
    all_results[candidate_name] = result

print("\n" + "═"*60)
print("  ✅ All candidates screened!")
print("  📊 LangSmith traces available at smith.langchain.com")
print("═"*60)

🚀 Starting AI Resume Screener...
   Job       : Senior Data Scientist @ TechCorp Analytics
   Candidates: 3
   LLM       : Groq / llama-3.1-8b-instant(free)
   Tracing   : LangSmith — project 'AI-Resume-Screener-Groq'

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Screening: strong_resume (Strong Candidate)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

⏳ Step 1/4: Extracting skills...
Languages : ['Python', 'SQL', 'R']
ML Tools  : ['Scikit-learn', 'XGBoost', 'LightGBM']
Cloud     : ['AWS (SageMaker, S3, EC2)', 'GCP']
Exp Years : 6

⏳ Step 2/4: Matching against JD...
 ✅ Matched : 12 skills — ['Python', 'SQL', 'Scikit-learn']
 ❌ Missing : 5 skills — ['TensorFlow or PyTorch', 'model evaluation', 'cross-validation']

⏳ Step 3/4: Computing score...
   ✅ Skills  : 40/50
   ✅ Exp     : 20/25
   ✅ Edu     : 10/10
   ✅ Bonus   : 8/10
   ✅ Achiev  : 4/5
   🎯 SCORE   : 82/100

⏳ Step 4/4: Generating explanation...

─────────────────────────────────────────────────

# Final Result

In [50]:
# ─────────────────────────────────────────────────────────────
# FINAL RESULTS TABLE
# Clean side-by-side comparison of all 3 candidates.
# ─────────────────────────────────────────────────────────────

print("\n" + "═"*70)
print("   🏆  AI RESUME SCREENER — FINAL RESULTS")
print("═"*70)
print(f"{'Candidate':<22} {'Expected':<20} {'Score':>7}  {'Label':<16} {'Correct?'}")
print(f"{'─'*22} {'─'*20} {'─'*7}  {'─'*16} {'─'*8}")

# Expected label mapping (for validation)
expected_labels = {
    "Strong Candidate"  : "STRONG",
    "Average Candidate" : "AVERAGE",
    "Weak Candidate"    : "WEAK"
}

for name, result in all_results.items():
    score    = result["final_score"]
    label    = result["label"]
    ctype    = result["candidate_type"]
    expected = expected_labels.get(ctype, "?")
    
    # Check if the AI label matches what we expect
    is_correct = "✅" if expected in label else "❌"
    
    print(f"{name:<22} {ctype:<20} {score:>5}/100  {label:<16} {is_correct}")

print("═"*70)

# Score breakdown table
print("\n📋 SCORE BREAKDOWN:")
print(f"{'─'*70}")
print(f"{'Candidate':<22} {'Skills/50':>10} {'Exp/25':>8} {'Edu/10':>8} {'Bonus/10':>10} {'Ach/5':>7}")
print(f"{'─'*22} {'─'*10} {'─'*8} {'─'*8} {'─'*10} {'─'*7}")

for name, result in all_results.items():
    sd = result["score_data"]
    print(f"{name:<22}"
          f" {str(sd.get('required_skills_score','?')):>10}"
          f" {str(sd.get('experience_score','?')):>8}"
          f" {str(sd.get('education_score','?')):>8}"
          f" {str(sd.get('nice_to_have_score','?')):>10}"
          f" {str(sd.get('achievements_score','?')):>7}")

print(f"{'─'*70}")

# Visual bar chart
print("\n📊 SCORE BARS:")
for name, result in all_results.items():
    score    = result["final_score"]
    label    = result["label"]
    bar_len  = int(score / 2)   # 100 pts → 50 chars max
    bar_char = "█" if score >= 75 else "▓" if score >= 45 else "░"
    bar      = bar_char * bar_len
    print(f"  {name:<22} {score:>3}/100 |{bar:<50}| {label}")

print()
print("Thresholds: 🟢 STRONG ≥75 | 🟡 AVERAGE ≥45 | 🔴 WEAK <45")


══════════════════════════════════════════════════════════════════════
   🏆  AI RESUME SCREENER — FINAL RESULTS
══════════════════════════════════════════════════════════════════════
Candidate              Expected               Score  Label            Correct?
────────────────────── ──────────────────── ───────  ──────────────── ────────
strong_resume          Strong Candidate        82/100  🟢 STRONG         ✅
average_resume         Average Candidate       45/100  🟡 AVERAGE        ✅
weak_resume            Weak Candidate          30/100  🔴 WEAK           ✅
══════════════════════════════════════════════════════════════════════

📋 SCORE BREAKDOWN:
──────────────────────────────────────────────────────────────────────
Candidate               Skills/50   Exp/25   Edu/10   Bonus/10   Ach/5
────────────────────── ────────── ──────── ──────── ────────── ───────
strong_resume                  40       20       10          8       4
average_resume                 30       10        0          

#  Debug Run (LangSmith Error Tracing)¶
A deliberately vague resume to test anti-hallucination rules.

In [51]:
# ─────────────────────────────────────────────────────────────
# DEBUG RUN — Intentionally flawed/vague resume
#
# Purpose:
#   Show that the anti-hallucination rules prevent score inflation.
#   A vague resume claiming "10 years" without specifics should
#   still receive a LOW score — proving the pipeline is robust.
#
# This run is tagged 'debug' in LangSmith.
# Find it at smith.langchain.com → filter by tag: debug
# ─────────────────────────────────────────────────────────────

@traceable(
    name="debug_vague_resume",
    tags=["debug", "groq", "intentional-error", "anti-hallucination-test"]
)
def run_debug_case():
    """Debug run testing anti-hallucination on vague resume input."""

    # ── BUGGY RESUME: Grand claims, zero specifics ────────────
    # This resume intentionally uses vague language to test
    # whether the extraction prompt correctly returns empty fields
    # rather than inventing skills from JD keywords.
    vague_resume = """
    Name: Test Candidate
    
    SUMMARY:
    I am a highly experienced data professional with 10 years of experience.
    I know all the latest technologies and have worked on many projects.
    I am a fast learner and passionate about data.
    
    SKILLS: Various technical skills in data and programming.
    
    EXPERIENCE:
    Data Role — Various Companies (2014–2024)
    - Worked on data projects using modern tools.
    
    EDUCATION: Completed some courses online.
    """

    print("\n⚠️  DEBUG RUN: Testing anti-hallucination rules")
    print("   Input: Vague resume with no specific skills mentioned")
    print("   Expected result: LOW score despite '10 years' claim")
    print("   LangSmith tag: 'debug'\n")

    # Step 1: Extract — should return mostly 'Not mentioned'
    print("[DEBUG] Step 1: Extraction...")
    raw     = extraction_chain.invoke({"resume": vague_resume})
    skills  = safe_parse_json(raw)
    print("[DEBUG] Extracted (should be sparse):")
    for k, v in skills.items():
        print(f"   {k:<28}: {v}")

    # Step 2: Match
    print("\n[DEBUG] Step 2: Matching...")
    raw_match = matching_chain.invoke({
        "extracted_skills" : json.dumps(skills, indent=2),
        "job_description"  : JOB_DESCRIPTION
    })
    match = safe_parse_json(raw_match)

    # Step 3: Score
    print("\n[DEBUG] Step 3: Scoring...")
    raw_score = scoring_chain.invoke({
        "match_analysis"   : json.dumps(match, indent=2),
        "extracted_skills" : json.dumps(skills, indent=2)
    })
    score = safe_parse_json(raw_score)
    final = score.get("final_score", 0)

    label = assign_label(final)
    print(f"\n[DEBUG] RESULT: {final}/100 → {label}")

    # Validate anti-hallucination
    if final <= 30:
        print("✅ Anti-hallucination PASSED — low score despite vague experience claims.")
        print("📸 Screenshot this trace in LangSmith as CORRECT debug evidence.")
    else:
        print("❌ Possible hallucination — score inflated from vague claims.")
        print("📸 Screenshot this trace in LangSmith to show the BUG.")
        print("🔍 Check which prompt step inflated the score.")

    return {"score": final, "label": label, "extracted": skills}


# Run debug case
debug_result = run_debug_case()

print("\n💡 To find this run in LangSmith:")
print("   smith.langchain.com → AI-Resume-Screener-Groq → filter tag: debug")


⚠️  DEBUG RUN: Testing anti-hallucination rules
   Input: Vague resume with no specific skills mentioned
   Expected result: LOW score despite '10 years' claim
   LangSmith tag: 'debug'

[DEBUG] Step 1: Extraction...
[DEBUG] Extracted (should be sparse):
   programming_languages       : []
   ml_frameworks               : []
   data_tools                  : []
   cloud_platforms             : []
   mlops_tools                 : []
   visualization_tools         : []
   years_of_experience         : 10
   education_level             : Not mentioned
   notable_achievements        : []
   domain_experience           : []

[DEBUG] Step 2: Matching...

[DEBUG] Step 3: Scoring...

[DEBUG] RESULT: 10/100 → 🔴 WEAK
✅ Anti-hallucination PASSED — low score despite vague experience claims.
📸 Screenshot this trace in LangSmith as CORRECT debug evidence.

💡 To find this run in LangSmith:
   smith.langchain.com → AI-Resume-Screener-Groq → filter tag: debug


In [52]:
# Export to JSON

In [53]:
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# EXPORT TO JSON
# Saves full pipeline outputs for all candidates to a JSON file.
# Useful for recruiter reports, audit trails, and debugging.
# ─────────────────────────────────────────────────────────────

report = {
    "metadata": {
        "title"            : "AI Resume Screening Report",
        "job_title"        : "Senior Data Scientist",
        "company"          : "TechCorp Analytics",
        "generated_at"     : datetime.now().isoformat(),
        "llm_used"         : "groq/llama3-8b-8192 (free)",
        "langsmith_project": os.environ.get("LANGCHAIN_PROJECT"),
        "label_thresholds" : {"strong": ">=75", "average": ">=45", "weak": "<45"}
    },
    "candidates": {}
}

# Pack each candidate's full pipeline output into the report
for name, result in all_results.items():
    report["candidates"][name] = {
        "expected_type"   : result["candidate_type"],
        "final_score"     : result["final_score"],
        "label"           : result["label"],
        "explanation"     : result["explanation"],
        "score_breakdown" : result["score_data"],
        "match_analysis"  : result["match_analysis"],
        "extracted_skills": result["extracted_skills"]
    }

# Write to file
output_path = "resume_screening_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"✅ Results exported to: {output_path}")
print(f"   Candidates : {len(report['candidates'])}")
print(f"   Generated  : {report['metadata']['generated_at']}")
print()

# Quick summary
print("Quick Summary:")
for name, data in report["candidates"].items():
    print(f"  {name:<22} → {data['final_score']:>3}/100  {data['label']}")

✅ Results exported to: resume_screening_results.json
   Candidates : 3
   Generated  : 2026-04-17T13:15:34.001872

Quick Summary:
  strong_resume          →  82/100  🟢 STRONG
  average_resume         →  45/100  🟡 AVERAGE
  weak_resume            →  30/100  🔴 WEAK
